# HAI Repeated Experiments (Colab Runner)

This notebook is designed to run the heavy pipeline in a Google Colab runtime from VS Code.

It executes:
1. run_multicollinearity_cleanup.py
2. run_repeated_experiments.py (20 seeds, checkpoint/resume enabled)

Setup behavior in Cell 2:
- Mounts Google Drive (Colab only)
- Auto-clones the public GitHub repository into /content/HAI_April_2026
- Falls back to manual path selection if needed

Run order recommendation:
- Run Cell 2 (project setup)
- Run Cell 3 (dependency install)
- Run Cell 4 (GPU/resource check and tuning setup)
- Continue with the remaining cells

Repository URL: https://github.com/ZahirAhmadChaudhry/HAI_April_2026

In [7]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/ZahirAhmadChaudhry/HAI_April_2026.git'
REPO_NAME = 'HAI_April_2026'

# Optional manual override:
# MANUAL_PROJECT_DIR = Path('/content/drive/MyDrive/HAI_April_2026')
MANUAL_PROJECT_DIR = None

clone_target = Path('/content') / REPO_NAME
if IN_COLAB and not clone_target.exists():
    print('Cloning public repository into', clone_target)
    try:
        subprocess.check_call(['git', 'clone', REPO_URL, str(clone_target)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Git clone failed. Check network or set MANUAL_PROJECT_DIR to an existing local copy.'
        ) from exc

candidates = [
    MANUAL_PROJECT_DIR,
    clone_target,
    Path('/content/drive/MyDrive/HAI_April_2026'),
    Path('/content/HAI_April_2026'),
    Path.cwd(),
]

project_dir = None
for c in candidates:
    if c is None:
        continue
    c = Path(c)
    if (c / 'clean_hai_dataset.csv').exists() and (c / 'run_repeated_experiments.py').exists():
        project_dir = c
        break

if project_dir is None:
    raise FileNotFoundError(
        'Could not find HAI_April_2026. Set MANUAL_PROJECT_DIR to the correct path.'
    )

os.chdir(project_dir)
print('Project directory:', project_dir)
print('Python executable:', sys.executable)
print('Current working directory:', Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/HAI_April_2026
Python executable: /usr/bin/python3
Current working directory: /content/HAI_April_2026


In [8]:
import subprocess
import sys

packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'imbalanced-learn',
    'optuna',
    'xgboost',
    'lightgbm',
    'catboost',
    'shap',
    'joblib',
    'matplotlib',
    'seaborn',
    'scipy'
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed.')

Dependencies installed.


In [9]:
import os
import subprocess
import traceback

def _run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()

cpu_threads = os.cpu_count() or 2
thread_vars = [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]
for var in thread_vars:
    os.environ[var] = str(cpu_threads)
os.environ["HAI_NUM_THREADS"] = str(cpu_threads)

gpu_detected = False
gpu_info = "No GPU detected via nvidia-smi"

rc, out, err = _run(["nvidia-smi", "-L"])
if rc == 0 and out:
    gpu_detected = True
    gpu_info = out.splitlines()[0]
    print("GPU detected:", gpu_info)
    rc2, out2, err2 = _run(["nvidia-smi"])
    if out2:
        print("\n=== nvidia-smi (truncated) ===")
        print("\n".join(out2.splitlines()[:20]))
else:
    print("GPU check:", gpu_info)
    if err:
        print("nvidia-smi error:", err)

if gpu_detected:
    # Quick smoke test to verify that GPU training actually works.
    try:
        import numpy as np
        from xgboost import XGBClassifier

        rng = np.random.default_rng(42)
        X = rng.normal(size=(256, 10))
        y = (X[:, 0] + X[:, 1] > 0).astype(int)

        model = XGBClassifier(
            n_estimators=20,
            max_depth=3,
            learning_rate=0.1,
            tree_method="hist",
            device="cuda",
            eval_metric="logloss",
            n_jobs=cpu_threads,
            random_state=42,
        )
        model.fit(X, y)
        print("XGBoost GPU smoke test: PASS")
    except Exception as exc:
        gpu_detected = False
        print("XGBoost GPU smoke test: FAIL, fallback to CPU.")
        print(str(exc))
        traceback.print_exc(limit=1)

os.environ["HAI_USE_GPU"] = "1" if gpu_detected else "0"
if gpu_detected and not os.environ.get("HAI_GPU_DEVICES", "").strip():
    os.environ["HAI_GPU_DEVICES"] = "0"

print("\nResource configuration ready:")
print("HAI_USE_GPU=", os.environ["HAI_USE_GPU"])
print("HAI_NUM_THREADS=", os.environ["HAI_NUM_THREADS"])
print("Thread env vars set to", cpu_threads)
print("This configuration will be used by run_repeated_experiments.py")

GPU detected: GPU 0: Tesla T4 (UUID: GPU-72ca7b2b-e746-0ea8-e537-414f3d864f38)

=== nvidia-smi (truncated) ===
Sat Apr  4 06:08:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                            

In [10]:
import subprocess
import sys

print('Running multicollinearity cleanup...')
subprocess.check_call([sys.executable, 'run_multicollinearity_cleanup.py'], cwd=project_dir)
print('Part 1 complete.')

Running multicollinearity cleanup...
Part 1 complete.


In [11]:
import json
import pandas as pd
from pathlib import Path

cleaned_path = Path(project_dir) / 'cleaned_feature_sets.json'
report_path = Path(project_dir) / 'multicollinearity_cleanup_report.md'
heatmap_path = Path(project_dir) / 'figures' / 'correlation_matrix.png'

payload = json.loads(cleaned_path.read_text(encoding='utf-8'))
print('Model A cleaned feature count:', len(payload['model_a_cleaned']))
print('Model B cleaned feature count:', len(payload['model_b_cleaned']))
print('Dropped features:', list(payload['dropped_features'].keys()))
print('Heatmap exists:', heatmap_path.exists())

print('\nReport preview:')
print('-' * 80)
print(report_path.read_text(encoding='utf-8')[:2000])

Model A cleaned feature count: 20
Model B cleaned feature count: 33
Dropped features: ['nurse_staffing_count', 'nurse_aide_staffing_count', 'nurse_anesthetist_staffing_count', 'total_staffing_count']
Heatmap exists: True

Report preview:
--------------------------------------------------------------------------------
# Multicollinearity Cleanup Report

Computed on training set only (2019+2020), after imputation and before encoding.

## 1. Correlation Pairs with |r| > 0.80
| feature_1 | feature_2 | r | abs_r |
| --- | --- | --- | --- |
| nurse_anesthetist_staffing_etp | nurse_anesthetist_staffing_count | 1 | 1 |
| nurse_aide_staffing_etp | nurse_aide_staffing_count | 0.999413 | 0.999413 |
| total_staffing_etp | total_staffing_count | 0.997995 | 0.997995 |
| nurse_aide_staffing_etp | total_staffing_count | 0.985932 | 0.985932 |
| nurse_aide_staffing_count | total_staffing_count | 0.984996 | 0.984996 |
| nurse_aide_staffing_etp | total_staffing_etp | 0.981851 | 0.981851 |
| nurse_staffing

In [12]:
import subprocess
import sys

# Set to True to run the full 20-seed repeated experiment (can take hours).
RUN_FULL_REPEATED_EXPERIMENT = True

if RUN_FULL_REPEATED_EXPERIMENT:
    print('Running repeated experiments (20 seeds, checkpoint-enabled)...')
    subprocess.check_call([sys.executable, 'run_repeated_experiments.py'], cwd=project_dir)
    print('Part 2 complete.')
else:
    print('Skipped Part 2. Set RUN_FULL_REPEATED_EXPERIMENT=True when ready.')

Running repeated experiments (20 seeds, checkpoint-enabled)...
Part 2 complete.


In [13]:
from pathlib import Path
import pandas as pd

expected_files = [
    'repeated_experiment_results.csv',
    'repeated_experiment_summary.md',
    'repeated_shap_stability.csv',
    'figures/auc_distribution_boxplot.png',
    'figures/auc_difference_histogram.png',
    'figures/shap_stability_plot.png'
]

missing = []
for rel in expected_files:
    p = Path(project_dir) / rel
    if not p.exists():
        missing.append(rel)

if missing:
    print('Missing outputs:')
    for m in missing:
        print('-', m)
else:
    print('All repeated-experiment outputs are present.')

results_path = Path(project_dir) / 'repeated_experiment_results.csv'
if results_path.exists():
    df = pd.read_csv(results_path)
    print('Results shape:', df.shape)
    print('Unique seeds:', sorted(df['seed'].unique().tolist())[:5], '...')
    print('Model rows per seed (should be 8):')
    print(df.groupby('seed').size().describe())
    display(df.head())

All repeated-experiment outputs are present.
Results shape: (160, 17)
Unique seeds: [43, 44, 45, 46, 47] ...
Model rows per seed (should be 8):
count    20.0
mean      8.0
std       0.0
min       8.0
25%       8.0
50%       8.0
75%       8.0
max       8.0
dtype: float64


,run_id,seed,model_id,feature_set,algorithm,n_features,cv_auc_roc,best_params_json,auc_roc,auc_pr,accuracy,precision,recall,specificity,f1,mcc,brier_score
0,1,43,Model_A_catboost,A,catboost,20,0.884914,"{""depth"": 7, ""iterations"": 490, ""l2_leaf_reg"":...",0.814244,0.698402,0.777778,0.700000,0.525,0.895349,0.600000,0.459402,0.168338
1,1,43,Model_A_lightgbm,A,lightgbm,20,0.871697,"{""colsample_bytree"": 0.6245962532396887, ""lear...",0.853198,0.764499,0.801587,0.702703,0.650,0.872093,0.675325,0.533619,0.134975
2,1,43,Model_A_random_forest,A,random_forest,20,0.878720,"{""class_weight"": null, ""max_depth"": 13, ""min_s...",0.864535,0.785974,0.825397,0.764706,0.650,0.906977,0.702703,0.584094,0.132322
3,1,43,Model_A_xgboost,A,xgboost,20,0.853238,"{""colsample_bytree"": 0.8792286178406238, ""gamm...",0.842151,0.754523,0.730159,0.553571,0.775,0.709302,0.645833,0.453683,0.171270
4,1,43,Model_B_catboost,B,catboost,33,0.879610,"{""depth"": 4, ""iterations"": 439, ""l2_leaf_reg"":...",0.775291,0.666526,0.761905,0.678571,0.475,0.895349,0.558824,0.414666,0.192660


## Notes

- run_repeated_experiments.py uses checkpointing via repeated_experiment_checkpoint.csv.
- If runtime disconnects, re-run the repeated experiment cell (now Cell 7); completed seeds are skipped automatically.
- The repository is public, so token setup is not required for cloning.
- Cell 4 validates GPU availability and configures CPU/GPU environment variables consumed by the experiment script.
- After completion, check repeated_experiment_summary.md for paper-ready aggregate results.

In [14]:
from pathlib import Path

expected_files = [
    "repeated_experiment_results.csv",
    "repeated_experiment_summary.md",
    "repeated_shap_stability.csv",
    "figures/auc_distribution_boxplot.png",
    "figures/auc_difference_histogram.png",
    "figures/shap_stability_plot.png",
]

print("project_dir =", project_dir)
root = Path(project_dir)

for rel in expected_files:
    p = root / rel
    print(("FOUND   " if p.exists() else "MISSING "), p)

project_dir = /content/HAI_April_2026
FOUND    /content/HAI_April_2026/repeated_experiment_results.csv
FOUND    /content/HAI_April_2026/repeated_experiment_summary.md
FOUND    /content/HAI_April_2026/repeated_shap_stability.csv
FOUND    /content/HAI_April_2026/figures/auc_distribution_boxplot.png
FOUND    /content/HAI_April_2026/figures/auc_difference_histogram.png
FOUND    /content/HAI_April_2026/figures/shap_stability_plot.png


In [15]:
from pathlib import Path
import shutil

expected_files = [
    "repeated_experiment_results.csv",
    "repeated_experiment_summary.md",
    "repeated_shap_stability.csv",
    "figures/auc_distribution_boxplot.png",
    "figures/auc_difference_histogram.png",
    "figures/shap_stability_plot.png",
]

src_root = Path(project_dir)
dst_root = Path("/content/drive/MyDrive/HAI_April_2026_outputs")
dst_root.mkdir(parents=True, exist_ok=True)

copied = []
for rel in expected_files:
    src = src_root / rel
    if src.exists():
        dst = dst_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied.append(str(dst))

print("Copied files:")
for p in copied:
    print("-", p)

Copied files:
- /content/drive/MyDrive/HAI_April_2026_outputs/repeated_experiment_results.csv
- /content/drive/MyDrive/HAI_April_2026_outputs/repeated_experiment_summary.md
- /content/drive/MyDrive/HAI_April_2026_outputs/repeated_shap_stability.csv
- /content/drive/MyDrive/HAI_April_2026_outputs/figures/auc_distribution_boxplot.png
- /content/drive/MyDrive/HAI_April_2026_outputs/figures/auc_difference_histogram.png
- /content/drive/MyDrive/HAI_April_2026_outputs/figures/shap_stability_plot.png


In [16]:
from pathlib import Path
import zipfile

expected_files = [
    "repeated_experiment_results.csv",
    "repeated_experiment_summary.md",
    "repeated_shap_stability.csv",
    "figures/auc_distribution_boxplot.png",
    "figures/auc_difference_histogram.png",
    "figures/shap_stability_plot.png",
]

root = Path(project_dir)
zip_path = root / "hai_repeated_outputs.zip"

added = []
missing = []

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in expected_files:
        src = root / rel
        if src.exists():
            zf.write(src, arcname=rel)
            added.append(rel)
        else:
            missing.append(rel)

print("Created:", zip_path)
print("Files packaged:", len(added))
for rel in added:
    print("  +", rel)

if missing:
    print("\nMissing and not packaged:")
    for rel in missing:
        print("  -", rel)

try:
    from google.colab import files
    files.download(str(zip_path))
    print("\nDownload started in browser.")
except Exception:
    print("\nNot running in Colab download context. Zip is available at:")
    print(zip_path)

Created: /content/HAI_April_2026/hai_repeated_outputs.zip
Files packaged: 6
  + repeated_experiment_results.csv
  + repeated_experiment_summary.md
  + repeated_shap_stability.csv
  + figures/auc_distribution_boxplot.png
  + figures/auc_difference_histogram.png
  + figures/shap_stability_plot.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download started in browser.
